# agents_4_puzzles: self-improvement + inline Kaggle auto-submit

Этот блокнот **не применяет patch/delta ZIP**. Он клонирует актуальный репозиторий через:

```bash
git clone --depth 1 https://github.com/visualcomments/agents_4_puzzles
```

и запускает базовый/продвинутый сценарии self-improvement. Дополнительно можно включить автосабмит успешного улучшенного `submission.csv` в Kaggle соревнование `cayley-py-megaminx` через inline `kaggle.json`.

⚠️ Не публикуйте блокнот с заполненным `KAGGLE_JSON_INLINE`: это секретный API-ключ. Для первого запуска лучше включить `KAGGLE_SUBMIT_DRY_RUN = True`.


In [ ]:
#@title 1. Конфигурация запуска
REPO_URL = "https://github.com/visualcomments/agents_4_puzzles"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
SCENARIO = "both"  #@param ["baseline", "basic", "advanced", "both", "all"]
COMPETITION = "cayley-py-megaminx"  #@param {type:"string"}
MODELS = "gpt-4o-mini"  #@param {type:"string"}
INSTALL_MODE = "llm"  #@param ["none", "min", "llm", "local", "full"]
SKIP_MODEL_PREFLIGHT = False  #@param {type:"boolean"}
SKIP_IMPROVED_FILE_CHECK = False  #@param {type:"boolean"}
FORCE_RECLONE = True  #@param {type:"boolean"}
ALLOW_FAILURES = True  #@param {type:"boolean"}

# Для smoke-теста поставьте небольшое число, например 20. Для полного прогона оставьте None.
MAX_ROWS = None  #@param {type:"raw"}
ROUNDS_BASIC = 3  #@param {type:"integer"}
ROUNDS_ADVANCED = 6  #@param {type:"integer"}
MAX_ITERS = 100000  #@param {type:"integer"}

# Kaggle auto-submit. Сначала используйте dry-run.
AUTO_SUBMIT_KAGGLE = False  #@param {type:"boolean"}
KAGGLE_SUBMIT_DRY_RUN = True  #@param {type:"boolean"}
KAGGLE_COMPETITION = "cayley-py-megaminx"  #@param {type:"string"}
SUBMIT_MIN_IMPROVEMENT = 1  #@param {type:"integer"}
SUBMIT_WITHOUT_BASELINE_IMPROVEMENT = False  #@param {type:"boolean"}
SUBMIT_ALL_SUCCESSFUL = False  #@param {type:"boolean"}
KAGGLE_SUBMIT_MESSAGE = "agents_4_puzzles {scenario} score={score} improvement={improvement} {timestamp}"  #@param {type:"string"}

# Вставьте содержимое kaggle.json целиком. Пример: {"username":"...","key":"..."}
# Безопаснее оставить пустым и заполнить через getpass в следующей ячейке.
KAGGLE_JSON_INLINE = r'''

'''.strip()
USE_GETPASS_FOR_KAGGLE_JSON = True  #@param {type:"boolean"}

WORKDIR = "/content/agents_4_puzzles_runs"
REPO_DIR = f"{WORKDIR}/agents_4_puzzles"
BASIC_VARIANT = "strict_self_improvement"
ADVANCED_VARIANT = "failure_aware_self_improvement"


In [ ]:
#@title 2. Вспомогательные функции
from __future__ import annotations

import csv
import datetime as dt
import json
import os
import shlex
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

EXPECTED_IMPROVED_FILES = [
    "pipeline_cli.py",
    "AgentLaboratory/perm_pipeline/run_perm_pipeline.py",
    "competitions/cayley-py-megaminx/prompt_self_improver.py",
    "competitions/cayley-py-megaminx/failure_aware_self_improvement/row_profile_memory.py",
    "competitions/cayley-py-megaminx/self_improvement_scenarios.py",
]

def now_stamp() -> str:
    return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def section(title: str):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

def run(cmd: Sequence[str], cwd: Optional[Path] = None, log_path: Optional[Path] = None, check: bool = True, env: Optional[Dict[str, str]] = None) -> int:
    printable = " ".join(shlex.quote(str(x)) for x in cmd)
    print(f"$ {'cd ' + str(cwd) + ' && ' if cwd else ''}{printable}", flush=True)
    log_fh = None
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_fh = log_path.open("w", encoding="utf-8", errors="replace")
        log_fh.write(f"$ {printable}\n")
        if cwd:
            log_fh.write(f"cwd={cwd}\n")
        log_fh.flush()
    proc = subprocess.Popen(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if log_fh:
            log_fh.write(line)
    rc = proc.wait()
    if log_fh:
        log_fh.write(f"\n[exit_code] {rc}\n")
        log_fh.close()
    if check and rc != 0:
        raise subprocess.CalledProcessError(rc, list(cmd))
    return int(rc)

def require_expected_files(repo_dir: Path, strict: bool = True) -> Dict[str, Any]:
    missing = [x for x in EXPECTED_IMPROVED_FILES if not (repo_dir / x).exists()]
    if missing and strict:
        raise FileNotFoundError("Missing expected improved files: " + ", ".join(missing))
    return {"expected": EXPECTED_IMPROVED_FILES, "missing": missing}

def moves_len(value: str) -> int:
    value = str(value or "").strip()
    if not value:
        return 0
    if value.upper() == "UNSOLVED":
        return 10**9
    if "." in value:
        return len([x for x in value.split(".") if x])
    return len([x for x in value.replace(",", " ").split() if x])

def score_submission(csv_path: Path) -> Dict[str, Any]:
    if not csv_path.exists():
        return {"exists": False, "path": str(csv_path)}
    total = row_count = max_len = 0
    col_name = None
    with csv_path.open(newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        fields = reader.fieldnames or []
        for candidate in ("path", "moves", "solution"):
            if candidate in fields:
                col_name = candidate
                break
        if col_name is None:
            return {"exists": True, "path": str(csv_path), "error": f"no path/moves/solution column in {fields}"}
        for row in reader:
            n = moves_len(row.get(col_name, ""))
            total += n
            max_len = max(max_len, n)
            row_count += 1
    return {"exists": True, "path": str(csv_path), "column": col_name, "rows": row_count, "total_move_tokens": total, "max_row_tokens": max_len}

def get_score(result: Dict[str, Any]) -> Optional[int]:
    score = result.get("score") or {}
    value = score.get("total_move_tokens") if isinstance(score, dict) else None
    return value if isinstance(value, int) and value < 10**9 else None

def install_kaggle_cli(repo_dir: Path, log_dir: Path):
    section("Install Kaggle CLI")
    run([sys.executable, "-m", "pip", "install", "-U", "kaggle"], cwd=repo_dir, log_path=log_dir / "pip_kaggle.log")

def setup_kaggle_credentials(raw: str, config_dir: Path) -> Dict[str, Any]:
    raw = (raw or "").strip()
    if not raw:
        raise ValueError("AUTO_SUBMIT_KAGGLE=True, но KAGGLE_JSON_INLINE пустой")
    try:
        payload = json.loads(raw)
        if isinstance(payload, str):
            payload = json.loads(payload)
    except json.JSONDecodeError as exc:
        raise ValueError("KAGGLE_JSON_INLINE не является валидным JSON") from exc
    if not isinstance(payload, dict) or not payload.get("username") or not payload.get("key"):
        raise ValueError('kaggle.json должен содержать поля "username" и "key"')
    config_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json_path = config_dir / "kaggle.json"
    kaggle_json_path.write_text(json.dumps({"username": payload["username"], "key": payload["key"]}), encoding="utf-8")
    try:
        os.chmod(config_dir, 0o700)
        os.chmod(kaggle_json_path, 0o600)
    except PermissionError:
        pass
    return {"configured": True, "config_dir": str(config_dir), "kaggle_json_path": str(kaggle_json_path), "username": payload["username"], "key_present": True}

def kaggle_cmd_prefix() -> List[str]:
    exe = shutil.which("kaggle")
    return [exe] if exe else [sys.executable, "-m", "kaggle"]

def select_submit_candidates(results: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    baseline_scores = [get_score(r) for r in results if str(r.get("scenario", "")).startswith("baseline") and get_score(r) is not None]
    baseline_best = min(baseline_scores) if baseline_scores else None
    candidates = []
    for r in results:
        scenario = str(r.get("scenario") or "")
        if scenario.startswith("baseline") or int(r.get("exit_code") or 0) != 0:
            continue
        score = get_score(r)
        output_csv = Path(str(r.get("output_csv") or ""))
        if score is None or not output_csv.exists():
            continue
        improvement = None if baseline_best is None else baseline_best - score
        eligible = False
        if baseline_best is not None:
            eligible = improvement is not None and improvement >= int(SUBMIT_MIN_IMPROVEMENT)
        elif SUBMIT_WITHOUT_BASELINE_IMPROVEMENT:
            eligible = True
        if eligible:
            candidates.append({"scenario": scenario, "score": score, "baseline_score": baseline_best, "improvement": improvement, "output_csv": str(output_csv)})
    candidates.sort(key=lambda x: (x["score"], x["scenario"]))
    selected = candidates if SUBMIT_ALL_SUCCESSFUL else candidates[:1]
    return {"baseline_score": baseline_best, "eligible_candidates": candidates, "selected_candidates": selected}

def submit_to_kaggle(repo_dir: Path, log_dir: Path, env: Dict[str, str], candidate: Dict[str, Any], index: int) -> Dict[str, Any]:
    scenario = candidate["scenario"]
    score = candidate["score"]
    improvement = candidate.get("improvement")
    message = KAGGLE_SUBMIT_MESSAGE.format(scenario=scenario, score=score, improvement=improvement, timestamp=now_stamp()) if KAGGLE_SUBMIT_MESSAGE else f"agents_4_puzzles {scenario} score={score} {now_stamp()}"
    cmd = kaggle_cmd_prefix() + ["competitions", "submit", KAGGLE_COMPETITION, "-f", candidate["output_csv"], "-m", message]
    label = f"{index:02d}_{scenario}".replace("/", "_").replace(" ", "_")
    log_path = log_dir / f"kaggle_submit_{label}.log"
    if KAGGLE_SUBMIT_DRY_RUN:
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
        print("[kaggle dry-run]", printable)
        log_path.write_text(f"[dry-run] {printable}\n", encoding="utf-8")
        return {"submitted": False, "dry_run": True, "competition": KAGGLE_COMPETITION, "candidate": candidate, "message": message, "log": str(log_path)}
    rc = run(cmd, cwd=repo_dir, env=env, log_path=log_path, check=False)
    return {"submitted": rc == 0, "dry_run": False, "exit_code": rc, "competition": KAGGLE_COMPETITION, "candidate": candidate, "message": message, "log": str(log_path)}


In [ ]:
#@title 3. Inline kaggle.json, если нужен автосабмит
if AUTO_SUBMIT_KAGGLE and USE_GETPASS_FOR_KAGGLE_JSON and not KAGGLE_JSON_INLINE.strip():
    KAGGLE_JSON_INLINE = getpass('Вставьте содержимое kaggle.json целиком: ')

print('AUTO_SUBMIT_KAGGLE =', AUTO_SUBMIT_KAGGLE)
print('KAGGLE_SUBMIT_DRY_RUN =', KAGGLE_SUBMIT_DRY_RUN)
print('KAGGLE_JSON_INLINE provided =', bool(KAGGLE_JSON_INLINE.strip()))


In [ ]:
#@title 4. Клонирование репозитория через git clone
workdir = Path(WORKDIR)
repo_dir = Path(REPO_DIR)
workdir.mkdir(parents=True, exist_ok=True)

section("Clone repository")
if FORCE_RECLONE and repo_dir.exists():
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        clone_cmd += ["--branch", BRANCH.strip()]
    clone_cmd += [REPO_URL, str(repo_dir)]
    run(clone_cmd)
else:
    print(f"Using existing checkout: {repo_dir}")

run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir)
run(["git", "status", "--short"], cwd=repo_dir)


In [ ]:
#@title 5. Установка зависимостей
run_root = workdir / f"self_improvement_run_{now_stamp()}"
log_dir = run_root / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

section(f"Install dependencies: {INSTALL_MODE}")
if INSTALL_MODE != "none":
    run([sys.executable, "-m", "pip", "install", "-U", "pip"], cwd=repo_dir, log_path=log_dir / "pip_upgrade.log")

requirements = []
if INSTALL_MODE in {"min", "llm", "local", "full"}:
    requirements.append(repo_dir / "requirements-min.txt")
if INSTALL_MODE in {"llm", "local", "full"}:
    requirements.append(repo_dir / "AgentLaboratory" / "requirements-llm.txt")
if INSTALL_MODE in {"local", "full"}:
    requirements.append(repo_dir / "AgentLaboratory" / "requirements-local-llm.txt")
if INSTALL_MODE == "full":
    requirements.append(repo_dir / "requirements-full.txt")

for req in requirements:
    if req.exists():
        run([sys.executable, "-m", "pip", "install", "-r", str(req)], cwd=repo_dir, log_path=log_dir / f"pip_{req.name}.log")
    else:
        print(f"Missing requirements file, skipped: {req}")


In [ ]:
#@title 6. Проверка файлов улучшенных сценариев и LLM/g4f preflight
section("Validate improved scenario files")
validation = require_expected_files(repo_dir, strict=not SKIP_IMPROVED_FILE_CHECK)
print(json.dumps(validation, ensure_ascii=False, indent=2))

py_files = [repo_dir / x for x in validation["expected"] if x.endswith(".py") and (repo_dir / x).exists()]
run([sys.executable, "-m", "py_compile", *map(str, py_files)], cwd=repo_dir, log_path=log_dir / "py_compile.log")

section("Model preflight")
model_preflight = {"skipped": SKIP_MODEL_PREFLIGHT}
if not SKIP_MODEL_PREFLIGHT:
    rc = run(
        [sys.executable, "pipeline_cli.py", "check-g4f-models", "--models", MODELS, "--timeout", "20", "--json"],
        cwd=repo_dir,
        log_path=log_dir / "model_preflight.log",
        check=False,
    )
    model_preflight["exit_code"] = rc
else:
    print("Skipped")


In [ ]:
#@title 7. Запуск сценариев basic/advanced и submit gate
summary = {
    "runner": "colab_no_patch_kaggle_inline",
    "repo_url": REPO_URL,
    "branch": BRANCH or None,
    "repo_dir": str(repo_dir),
    "run_root": str(run_root),
    "competition": COMPETITION,
    "scenario": SCENARIO,
    "models": MODELS,
    "rounds_basic": ROUNDS_BASIC,
    "rounds_advanced": ROUNDS_ADVANCED,
    "max_iters": MAX_ITERS,
    "max_rows": MAX_ROWS,
    "patch_applied": False,
    "validation": validation,
    "model_preflight": model_preflight,
    "kaggle_auto_submit_enabled": AUTO_SUBMIT_KAGGLE,
    "kaggle_competition": KAGGLE_COMPETITION,
    "kaggle_submit_dry_run": KAGGLE_SUBMIT_DRY_RUN,
    "submit_min_improvement": SUBMIT_MIN_IMPROVEMENT,
    "submit_without_baseline_improvement": SUBMIT_WITHOUT_BASELINE_IMPROVEMENT,
    "submit_all_successful": SUBMIT_ALL_SUCCESSFUL,
    "started_at": dt.datetime.now().isoformat(timespec="seconds"),
    "results": [],
}

def run_scenario(name: str, prompt_variant: Optional[str], rounds: int, extra_args: Sequence[str], no_llm: bool = False):
    section(f"Run scenario: {name}")
    out_dir = run_root / name
    out_dir.mkdir(parents=True, exist_ok=True)
    output_csv = out_dir / f"{name}_submission.csv"
    run_log = out_dir / f"{name}_run_log.json"
    stdout_log = log_dir / f"{name}.stdout.log"
    cmd = [
        sys.executable, "pipeline_cli.py", "run",
        "--competition", COMPETITION,
        "--output", str(output_csv),
        "--run-log", str(run_log),
        "--no-progress", "--schema-check",
    ]
    if no_llm:
        cmd.append("--no-llm")
    else:
        cmd += [
            "--models", MODELS,
            "--keep-improving",
            "--self-improve-prompts",
            "--reject-identical-candidates",
            "--write-per-row-delta",
            "--improvement-rounds", str(rounds),
            "--max-iters", str(MAX_ITERS),
            "--g4f-stop-at-python-fence",
        ]
        if prompt_variant:
            cmd += ["--prompt-variant", prompt_variant]
    if MAX_ROWS is not None and int(MAX_ROWS) > 0:
        cmd += ["--max-rows", str(MAX_ROWS)]
    cmd += list(extra_args)
    rc = run(cmd, cwd=repo_dir, log_path=stdout_log, check=False)
    result = {
        "scenario": name,
        "exit_code": rc,
        "output_csv": str(output_csv),
        "run_log": str(run_log),
        "stdout_log": str(stdout_log),
        "score": score_submission(output_csv),
    }
    if run_log.exists():
        try:
            result["run_log_json"] = json.loads(run_log.read_text(encoding="utf-8", errors="replace"))
        except Exception as exc:
            result["run_log_parse_error"] = repr(exc)
    summary["results"].append(result)
    return result

if SCENARIO in {"baseline", "all"}:
    run_scenario("baseline", None, 0, [], no_llm=True)

if SCENARIO in {"basic", "both", "all"}:
    run_scenario("basic_strict_self_improvement", BASIC_VARIANT, ROUNDS_BASIC, [], no_llm=False)

if SCENARIO in {"advanced", "both", "all"}:
    advanced_extra = ["--search-mode", "hybrid", "--plan-beam-width", "4", "--frontier-width", "8", "--archive-size", "12", "--refine-rounds", "2"]
    run_scenario("advanced_failure_aware_self_improvement", ADVANCED_VARIANT, ROUNDS_ADVANCED, advanced_extra, no_llm=False)

# Если автосабмит включён, по умолчанию сначала измеряем baseline, чтобы не отправлять no-op.
if AUTO_SUBMIT_KAGGLE and not SUBMIT_WITHOUT_BASELINE_IMPROVEMENT:
    has_baseline = any(str(r.get("scenario") or "").startswith("baseline") for r in summary["results"])
    if not has_baseline:
        run_scenario("baseline_for_submit", None, 0, [], no_llm=True)

if AUTO_SUBMIT_KAGGLE:
    section("Kaggle auto-submit gate")
    selection = select_submit_candidates(summary["results"])
    summary["kaggle_auto_submit"] = {"enabled": True, "selection": selection, "submissions": []}
    if not selection["selected_candidates"]:
        print("No eligible improved successful submission found; nothing to submit.")
        summary["kaggle_auto_submit"]["skipped_reason"] = "no eligible improved successful submission found"
    else:
        install_kaggle_cli(repo_dir, log_dir)
        cred_meta = setup_kaggle_credentials(KAGGLE_JSON_INLINE, run_root / ".kaggle")
        summary["kaggle_auto_submit"]["credentials"] = cred_meta
        env = os.environ.copy()
        env["KAGGLE_CONFIG_DIR"] = cred_meta["config_dir"]
        for i, candidate in enumerate(selection["selected_candidates"], start=1):
            summary["kaggle_auto_submit"]["submissions"].append(submit_to_kaggle(repo_dir, log_dir, env, candidate, i))
else:
    summary["kaggle_auto_submit"] = {"enabled": False}

summary["finished_at"] = dt.datetime.now().isoformat(timespec="seconds")
summary_path = run_root / "self_improvement_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
section("Summary")
print(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True))
print(f"Summary saved to: {summary_path}")

failed = [r for r in summary["results"] if r.get("exit_code") != 0]
if failed and not ALLOW_FAILURES:
    raise RuntimeError(f"{len(failed)} scenario(s) failed; see logs under {log_dir}")


In [ ]:
#@title 8. Упаковка артефактов прогона в ZIP
section("Zip run artifacts")
zip_path = shutil.make_archive(str(run_root), "zip", root_dir=run_root)
print(f"ZIP: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print(f"Download helper is unavailable outside Colab: {exc}")
